# market_neural_net — Colab training launcher

Reusable GPU launcher shell per the project README (§2.1: local CPU handles data/backtesting, this notebook handles GPU-hungry training).

Code comes straight from GitHub (https://github.com/yuvidewan/market_neural_net) — no manual zip/upload step needed for that anymore.

**One-time setup, since the repo is private:** a plain `git clone` fails in Colab with `fatal: could not read Username for 'https://github.com'` because there's nothing to authenticate with. Fix, once:
1. GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → generate one scoped to just this repo, read-only "Contents" permission.
2. In Colab, click the 🔑 key icon in the left sidebar → Add secret → name it `GITHUB_TOKEN`, paste the token → toggle "Notebook access" on.

Cell 2 below reads that secret automatically (never prints it, and scrubs the token back out of the repo's local git config right after cloning/pulling). If no secret is found it falls back to a plain clone, which only works if you ever make the repo public.

Only the curated **data** needs a one-time Drive upload (it stays out of git on purpose — see `.gitignore`):
- Run `python -m scripts.package_for_colab --skip-data` locally if you ever want the old zip-based code path instead (e.g. offline, or before you've pushed a change) — not needed for normal use now.
- Upload `experiments/colab_data_bundle.zip` (~350MB, produced by the same script) to `My Drive/market_neural_net/colab_data_bundle.zip` — once, and again only after a real re-ingest of the curated dataset.

**Checkpoint/resume exists** (per-epoch, per-fold — `src/train/checkpointing.py`). Design, after a real failure taught us the naive version: `--out-dir` stays on the Colab VM's **local disk** (fast, reliable), and `--mirror-dir` points at Drive as a **best-effort backup** — every real-run cell below passes both. Local-first matters because Drive's FUSE mount doesn't handle interrupted writes reliably; an earlier version that wrote checkpoints straight to Drive could get its resume state corrupted by a write cut off mid-way, which crashed the *next* resume attempt too — the opposite of what checkpointing is for. The current design survives that: local writes are always reliable within a session, and if local disk gets wiped by a full VM disconnect/recycle, the next run auto-hydrates from whatever the Drive mirror last caught up to. Pass `--fresh` on a cell's command to discard existing checkpoints (local and mirrored) and start that run over. The smoke-test cells don't bother with any of this — cheap to lose, not worth it.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned — Runtime > Change runtime type > GPU, then re-run this cell.')

## 2. Clone the repo

In [ ]:
REPO_SLUG = 'yuvidewan/market_neural_net'
PLAIN_URL = f'https://github.com/{REPO_SLUG}.git'
PROJECT_DIR = '/content/market_neural_net'

import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

if not token:
    print('No GITHUB_TOKEN Colab secret found -- this only works if the repo is public.')
    print('For a private repo: GitHub > Settings > Developer settings > Personal access')
    print('tokens > Fine-grained tokens > generate one, read-only, scoped to this repo.')
    print('Then in Colab: key icon (left sidebar) > Add secret > name it GITHUB_TOKEN >')
    print('paste the token > enable notebook access > re-run this cell.')

auth_url = f'https://{token}@github.com/{REPO_SLUG}.git' if token else PLAIN_URL

if os.path.exists(PROJECT_DIR):
    %cd $PROJECT_DIR
    !git remote set-url origin $auth_url
    !git pull
    !git remote set-url origin $PLAIN_URL   # scrub the token back out of .git/config
else:
    !git clone $auth_url $PROJECT_DIR
    %cd $PROJECT_DIR
    !git remote set-url origin $PLAIN_URL   # scrub the token back out of .git/config

## 3. Mount Drive and unpack the curated data
Data stays out of git (see `.gitignore`) — this is the one thing that still needs a manual Drive upload.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/My Drive/market_neural_net'  # change if you uploaded elsewhere

In [ ]:
import zipfile, os

data_zip = f'{DRIVE_PROJECT_DIR}/colab_data_bundle.zip'
assert os.path.exists(data_zip), (
    f'missing {data_zip} — run `python -m scripts.package_for_colab` locally and upload '
    f'experiments/colab_data_bundle.zip to that Drive path first'
)
with zipfile.ZipFile(data_zip) as zf:
    zf.extractall(PROJECT_DIR)
print('data extracted into', PROJECT_DIR)

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Sanity checks: tests, then tiny smoke-test runs
Per README §2.1 rule 1 — always verify the tiny config before trusting a real run on a new machine.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# M2 tiny smoke test — should finish in well under a minute even on CPU.
!python -u -m scripts.train_lstm_baseline \
  --n-symbols 5 --seq-len 20 --hidden-size 16 --num-layers 1 --epochs 1 \
  --test-years 2024 --out-dir experiments/lstm_baseline_smoketest

In [ ]:
# M3 tiny smoke test (TCN + quantile regression) — should also finish in under a minute.
!python -u -m scripts.train_ssl_quantile \
  --n-symbols 5 --seq-len 20 --channels 16 --epochs 1 \
  --test-years 2024 --out-dir experiments/ssl_quantile_smoketest

In [ ]:
# v2 transformer tiny smoke test -- should also finish in under a minute.
!python -u -m scripts.train_transformer_ssl \
  --n-symbols 5 --seq-len 16 --d-model 16 --n-heads 2 --n-blocks 1 --patch-size 4 --epochs 1 \
  --test-years 2024 --out-dir experiments/transformer_smoketest

## 6. M2 real run: LSTM baseline
Full README-spec baseline: 40 liquid symbols, seq_len=120, hidden=128, 2-layer LSTM, 4 walk-forward folds. On a T4 this should be well under the local-CPU runtime (a local CPU run of this exact config took 6.5 hours). Already run once locally — rerun here mainly to confirm parity, or after a code change.

`-u` keeps stdout unbuffered so progress prints as it happens instead of only at the end.

In [ ]:
!python -u -m scripts.train_lstm_baseline \
  --n-symbols 40 --seq-len 120 --hidden-size 128 --num-layers 2 --epochs 6 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/lstm_baseline \
  --mirror-dir "{DRIVE_PROJECT_DIR}/experiments/lstm_baseline"

## 7. M3 real run: TCN + self-supervised quantile regression
**This is the one that actually needs the GPU** — the M3 gate (README): out-of-sample cross-sectional rank IC > 0.02, stable sign across all 4 walk-forward folds. TCN convolutions parallelize across time (unlike the LSTM's sequential recurrence), so this should scale to a meaningfully larger universe than M2's 40 symbols without another multi-hour surprise — 200 liquid names here, big enough that the daily cross-sectional rank correlation is measuring something statistically real rather than noise from a handful of names.

The report includes the M3 gate PASS/FAIL verdict directly — read the per-fold breakdown either way, a single overall number hides whether one fold is carrying the result.

In [ ]:
!python -u -m scripts.train_ssl_quantile \
  --n-symbols 200 --seq-len 120 --channels 64 --epochs 8 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/ssl_quantile_tcn \
  --mirror-dir "{DRIVE_PROJECT_DIR}/experiments/ssl_quantile_tcn"

## 8. v2 real run: two-axis Transformer (README's "main model")
Same gate as M3 (mean OOS rank IC > 0.02, stable sign), same universe/fold setup, same metrics — the point is a direct comparison against the TCN's already-passing M3 result (0.0252 overall IC). The only things that differ are the encoder (two-axis attention vs. TCN) and how data is batched (one cross-sectional panel per date, since cross-sectional attention needs every symbol at once — see `src/data/panel_dataset.py`).

This one does far more optimizer steps than the TCN run (one per calendar date rather than one per fixed-size batch of independent samples), so it's the one most likely to actually take a while — and the one checkpoint/resume matters most for. It's covered: local `--out-dir` + Drive `--mirror-dir` (see the intro cell), so a disconnect just means re-running this same cell. Start with a partial universe/fewer epochs if you want a faster first read before committing to the full README-spec run below.

In [ ]:
!python -u -m scripts.train_transformer_ssl \
  --n-symbols 200 --seq-len 120 --d-model 256 --n-heads 8 --n-blocks 8 --patch-size 16 --epochs 8 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/ssl_quantile_transformer \
  --mirror-dir "{DRIVE_PROJECT_DIR}/experiments/ssl_quantile_transformer"

## 9. Sync results back to Drive
`--mirror-dir` only backs up checkpoints/resume-state as training goes — the final `report.json` itself is written once, locally, at the very end of a run. Run this cell after a run completes (or any time you want a snapshot) to get everything, including the final reports, onto Drive.

In [ ]:
import shutil
dest = f'{DRIVE_PROJECT_DIR}/experiments_from_colab'
shutil.copytree('experiments', dest, dirs_exist_ok=True)
print('synced experiments/ ->', dest)